# Graeco-Latin Square Experiments for Euler Square

This notebook documents step by step attempts to find or generate valid Graeco-Latin-style solutions used as guide layouts in the Euler Square game.

The puzzle rule we check is:
- In each row and column, outer symbols are unique.
- In each row and column, inner symbols are unique.
- Ordered pairs (outer, inner) are unique across the whole n x n board.

## 1. Validation Helpers

We begin with validators so every experiment can be checked by the same criteria.

In [ ]:
# pylint: disable=redefined-outer-name
"""Validation helpers for orthogonal Latin squares (Euler squares).
The outer and inner squares are represented as 2D lists of integers
in the range 0..n-1.
"""
from __future__ import annotations


def validate_pair_square(outer_square, inner_square):
    """Validate a pair of orthogonal Latin squares (Euler square).

    Args:
        outer_square: 2D list of integers for the outer square.
        inner_square: 2D list of integers for the inner square.

    Returns:
        Tuple (is_valid, message).
    """
    order = len(outer_square)
    assert order > 0 and all(
        len(row) == order for row in outer_square
    ), "outer must be n x n"
    assert (
        len(inner_square) == order
        and all(len(row) == order for row in inner_square)
    ), "inner must be n x n"

    symbols = set(range(order))

    # Row uniqueness
    for row_idx in range(order):
        if set(outer_square[row_idx]) != symbols:
            return (
                False,
                f"outer row {row_idx} not a permutation of 0..{order - 1}",
            )
        if set(inner_square[row_idx]) != symbols:
            return (
                False,
                f"inner row {row_idx} not a permutation of 0..{order - 1}",
            )

    # Column uniqueness
    for col_idx in range(order):
        col_outer = {outer_square[row_idx][col_idx] for row_idx in range(order)}
        col_inner = {inner_square[row_idx][col_idx] for row_idx in range(order)}
        if col_outer != symbols:
            return (
                False,
                f"outer column {col_idx} not a permutation of 0..{order - 1}",
            )
        if col_inner != symbols:
            return (
                False,
                f"inner column {col_idx} not a permutation of 0..{order - 1}",
            )

    # Orthogonality: all ordered pairs must be unique.
    pairs = {
        (outer_square[row_idx][col_idx], inner_square[row_idx][col_idx])
        for row_idx in range(order)
        for col_idx in range(order)
    }
    if len(pairs) != order * order:
        return False, f"pair uniqueness fails: {len(pairs)} != {order * order}"

    return True, "valid"

## 2. Fast Construction for Odd n

A simple affine construction works for odd n:
- Outer: $(r + c) \bmod n$
- Inner: $(r + 2c) \bmod n$

This is exactly why 3x3, 5x5, 7x7, ... can be generated instantly with deterministic code.

In [ ]:
# pylint: disable=redefined-outer-name
def construct_odd_affine(order):
    """Construct an Euler square pair for odd order using affine formulas.

    Args:
        order: Odd order of the Latin squares.

    Returns:
        Tuple (outer_square, inner_square).
    """
    assert order % 2 == 1
    outer_square = [
        [(row_idx + col_idx) % order for col_idx in range(order)]
        for row_idx in range(order)
    ]
    inner_square = [
        [(row_idx + 2 * col_idx) % order for col_idx in range(order)]
        for row_idx in range(order)
    ]
    return outer_square, inner_square


for order in [1, 3, 5, 7, 9]:
    outer_square, inner_square = construct_odd_affine(order)
    is_valid, message = validate_pair_square(outer_square, inner_square)
    print(f"n={order}: {is_valid} ({message})")

## 3. Why n=2 and n=6 Are Special

From known theory for Graeco-Latin squares (orthogonal Latin squares):
- n = 2: impossible
- n = 6: impossible

So in the app, showing a question-mark guide for these sizes is a principled fallback.

## 4. Search-Based Solver (Didactic, Not Production)

To understand complexity, we can run a constrained backtracking search.

Notes from experiments:
- Works on very small n, but branching explodes quickly.
- Even with row/column pruning, n grows hard fast.
- Deterministic constructions are far better for runtime UI guides.

In [ ]:
# pylint: disable=redefined-outer-name,too-many-locals
from itertools import permutations
from random import Random
from time import perf_counter


def latin_base(order):
    """Construct a base Latin square of the given order."""
    return [
        [(row_idx + col_idx) % order for col_idx in range(order)]
        for row_idx in range(order)
    ]


def try_build_orthogonal_by_row_perm(
    order, seed=0, max_tries=20000
):
    """Try to build an orthogonal pair via row-wise permutations.

    Args:
        order: Order of the Latin squares.
        seed: Random seed for reproducibility.
        max_tries: Maximum number of attempts.

    Returns:
        Tuple (result, elapsed_time, steps).
    """
    rng = Random(seed)
    outer_square = latin_base(order)

    # Build inner as row-wise permutations of symbols.
    perms = list(permutations(range(order)))
    rng.shuffle(perms)

    # Column symbol usage for inner square.
    col_used = [set() for _ in range(order)]
    pair_used = set()
    inner_square = [None] * order

    def dfs(row_idx, budget):
        if budget[0] <= 0:
            return False
        if row_idx == order:
            return True

        rng.shuffle(perms)
        for perm in perms:
            budget[0] -= 1
            good = True
            local_pairs = []

            for col_idx in range(order):
                symbol = perm[col_idx]
                if symbol in col_used[col_idx]:
                    good = False
                    break
                pair = (outer_square[row_idx][col_idx], symbol)
                if pair in pair_used:
                    good = False
                    break
                local_pairs.append((col_idx, symbol, pair))

            if not good:
                continue

            inner_square[row_idx] = list(perm)
            for col_idx, symbol, pair in local_pairs:
                col_used[col_idx].add(symbol)
                pair_used.add(pair)

            if dfs(row_idx + 1, budget):
                return True

            for col_idx, symbol, pair in local_pairs:
                col_used[col_idx].remove(symbol)
                pair_used.remove(pair)
            inner_square[row_idx] = None

        return False

    budget = [max_tries]
    t0 = perf_counter()
    found = dfs(0, budget)
    t1 = perf_counter()

    if not found:
        return None, t1 - t0, max_tries - budget[0]

    is_valid, message = validate_pair_square(outer_square, inner_square)
    result = (outer_square, inner_square, is_valid, message)
    return result, t1 - t0, max_tries - budget[0]


for order in [3, 4, 5, 6]:
    result, elapsed, steps = try_build_orthogonal_by_row_perm(
        order, seed=7, max_tries=40000
    )
    if result is None:
        print(
            f"n={order}: no solution found within budget, "
            f"time={elapsed:.3f}s, steps={steps}"
        )
    else:
        _, _, is_valid, message = result
        print(
            f"n={order}: found={is_valid}, msg={message}, "
            f"time={elapsed:.3f}s, steps={steps}"
        )

## 5. Implications for the App

### What worked well
- Deterministic odd-n affine construction is immediate and stable.
- n=2 and n=6 can be flagged early with explicit no-solution guide behavior.

### What is complex
- Generic search has large branching factor and sensitivity to heuristics.
- Even n (other than 2, 6) are solvable mathematically, but a robust lightweight constructor is less trivial than odd n.
- UI runtime generation should avoid heavy search loops to keep interaction smooth.

### Recommended engineering path
1. Keep deterministic constructors for odd n.
2. Add deterministic constructors or precomputed templates for even solvable n in your range.
3. Keep no-solution guard for n=2 and n=6 with the question-mark guide overlay.